# 05b: Computational efficiency, GPU benchmark (Colab T4)

**Paper:** Section *Computational Efficiency* (`sec:exp:efficiency`), paragraphs *Convergence* and *Cost at matched accuracy*. We compute our GPU numbers on a Google Colab Tesla T4 (torch 2.14.1+cu130).

The DIWECT (`diwect.py`) is compared with pyECT's WECT averaged over $d$ directions. The float64 DIWECT is used as the exact reference, and errors are the maximum absolute error relative to the maximum of the reference.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = Path("/content").is_dir()
WORK = Path(os.environ.get("BENCH_DIR", "/content" if IN_COLAB else "runs/efficiency"))
INPUTS = WORK / "inputs.npz" if IN_COLAB else Path("data/efficiency/inputs.npz")
WORK.mkdir(parents=True, exist_ok=True)
SMOKE = False
DIWECT_ONLY = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyect", "trimesh"], check=True)

import numpy as np
import torch
from pyect import WECT, sample_directions_2d, sample_directions_3d

from diwect import DEFAULT_CHUNK_SIZE, DIWECT

CHUNK = dict(DEFAULT_CHUNK_SIZE)
if (WORK / "chunk.json").exists():
    CHUNK.update({int(k): v for k, v in json.load(open(WORK / "chunk.json")).items()})

GPU = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA = GPU.type == "cuda"
HEIGHTS = (17, 65, 257)
DIRS = [2 ** k for k in range(3, 9 if SMOKE else 17)]
WECT_TIME_CAP = 30.0
WECT_BATCH_BYTES = 3 * 2 ** 30
RESULTS = WORK / ("results_diwect.jsonl" if DIWECT_ONLY else "results.jsonl")
PREVIOUS = WORK / "results.jsonl"
print(f"device {GPU}, chunk sizes {CHUNK}, writing {RESULTS}")

## Resume, measurement helpers, inputs

In [ ]:
DONE, kept = set(), []
if RESULTS.exists() and not DIWECT_ONLY:
    old = [json.loads(line) for line in open(RESULTS) if line.strip()]
    DONE = {r["complex"] for r in old if r["kind"] == "wect_h_check" and r["h"] == 257}
    kept = [r for r in old if r.get("complex") in DONE or r["kind"] == "env"]
    with open(RESULTS, "w") as f:
        f.writelines(json.dumps(r) + "\n" for r in kept)
OUT = open(RESULTS, "w" if DIWECT_ONLY else "a")


def emit(**rec):
    OUT.write(json.dumps(rec) + "\n")
    OUT.flush()
    print(json.dumps(rec), flush=True)


def sync():
    if CUDA:
        torch.cuda.synchronize()


def measure(fn, reps, warmup=True):
    """(median seconds, peak extra GPU bytes, output) of fn() after one warm-up."""
    out = fn() if warmup else None
    sync()
    times = []
    if CUDA:
        torch.cuda.empty_cache()
        base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
    for _ in range(reps):
        sync()
        start = time.perf_counter()
        out = fn()
        sync()
        times.append(time.perf_counter() - start)
    peak = torch.cuda.max_memory_allocated() - base if CUDA else 0
    return float(np.median(times)), int(peak), out


def reps_for(seconds):
    return 1 if seconds > 5 else 3 if seconds > 0.5 else 5


data = np.load(INPUTS)
index = json.loads(bytes(data["index"]).decode())


def load(name, dtype):
    cx = []
    for k in range(len(index[name]["cells"])):
        x = torch.as_tensor(data[f"{name}/{k}/x"])
        x = x.to(GPU, dtype) if k == 0 else x.to(GPU)
        cx.append((x, torch.as_tensor(data[f"{name}/{k}/w"]).to(GPU, dtype)))
    return cx


def random_directions(d, dim, seed):
    g = torch.Generator(device="cpu").manual_seed(seed)
    v = torch.randn(d, dim, generator=g, dtype=torch.float64)
    return (v / v.norm(dim=1, keepdim=True)).float().to(GPU)


def wect_mean(cx, dirs, h, batch):
    """Average of pyECT's WECT over dirs, in batches of `batch` directions (float64 accumulator)."""
    acc = torch.zeros(h, dtype=torch.float64, device=GPU)
    for i in range(0, dirs.size(0), batch):
        acc += WECT(dirs[i:i + batch], h)(cx).double().sum(0)
    return acc / dirs.size(0)


def batch_size(name):
    # pyECT materialises v_indices[simp_verts]: (k_i, i+1, d) int64, plus a (k_i, d) max
    per_dir = max((k + 2) * n for k, n in enumerate(index[name]["cells"])) * 8 * 1.5
    return max(1, int(WECT_BATCH_BYTES // per_dir))


def rel_err(approx, exact):
    return float((approx - exact).abs().max() / exact.abs().max())

## Environment record and reference values

In [ ]:
if CUDA:
    props = torch.cuda.get_device_properties(0)
    emit(kind="env", torch=torch.__version__, cuda=torch.version.cuda, gpu=props.name,
         gpu_mem_gb=props.total_memory / 2 ** 30, chunk=CHUNK)
else:
    emit(kind="env", torch=torch.__version__, device="cpu", chunk=CHUNK)

previous = {}
if DIWECT_ONLY and PREVIOUS.exists():
    for line in open(PREVIOUS):
        r = json.loads(line)
        if r["kind"] == "diwect" and r.get("values"):
            previous[r["complex"], r["h"]] = torch.tensor(r["values"], dtype=torch.float64, device=GPU)

references = {}
for r in kept:
    if r["kind"] == "diwect" and r["dtype"] == "float64" and "values" in r:
        references[r["complex"], r["h"]] = torch.tensor(r["values"], dtype=torch.float64, device=GPU)


def reference(name, h):
    """float64 DIWECT of `name` at h heights"""
    if (name, h) not in references:
        references[name, h] = DIWECT(h, normalize=True, chunk_size=CHUNK[index[name]["dim"]])(
            load(name, torch.float64)).detach()
    return references[name, h]

## The benchmark

In [ ]:
order = sorted(index, key=lambda n: (index[n]["kind"] == "cubical", index[n]["total_cells"]))
if SMOKE:
    order = [n for n in order if index[n.replace("cubical", "freudenthal")]["total_cells"] < 20000]
if DONE:
    print(f"resuming: skipping {len(DONE)} finished complexes", flush=True)
for name in order:
    if name in DONE:
        continue
    meta = index[name]
    dim = meta["dim"]
    common = dict(complex=name, cells=meta["total_cells"], ctype=meta["kind"], dim=dim, source=meta["source"])
    cx64, cx32 = load(name, torch.float64), load(name, torch.float32)

    # --- DIWECT ------------------------------------------------------------- #
    ref_name = name.replace("cubical", "freudenthal")
    if meta["kind"] != "cubical" or DIWECT_ONLY:
        for h in HEIGHTS:
            for dtype, cx in ((torch.float64, cx64), (torch.float32, cx32)):
                mod = DIWECT(h, normalize=True, dtype=dtype, chunk_size=CHUNK[dim])
                try:
                    t, _, _ = measure(lambda: mod(cx), 1)
                    t, mem, out = measure(lambda: mod(cx), reps_for(t))
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    emit(kind="diwect", h=h, dtype=str(dtype)[6:], oom=True, chunk_size=CHUNK[dim], **common)
                    continue
                if dtype == torch.float64 and meta["kind"] != "cubical":
                    references[name, h] = out.detach()
                err = rel_err(out.double(), reference(ref_name, h))
                extra = {"values": out.tolist()} if dtype == torch.float64 else {}
                if dtype == torch.float64 and (name, h) in previous:
                    extra["rel_err_vs_previous"] = rel_err(out, previous[name, h])
                emit(kind="diwect", h=h, dtype=str(dtype)[6:], seconds=t, peak_bytes=mem, rel_err=err,
                     chunk_size=CHUNK[dim], **extra, **common)
    if DIWECT_ONLY:
        del cx64, cx32
        if CUDA:
            torch.cuda.empty_cache()
        continue
    if meta["kind"] == "cubical":
        same = bool(np.array_equal(data[f"{name}/0/x"], data[f"{ref_name}/0/x"]))
        emit(kind="note", msg=f"{name}: reference {ref_name}, identical vertices: {same}", **common)
        # WECT of the cubical complex vs. its Freudenthal triangulation, same directions
        sample = sample_directions_2d if dim == 2 else sample_directions_3d
        mod = WECT(sample(64, device=GPU), 65)
        a = mod(cx32).double()
        b = mod(load(ref_name, torch.float32)).double()
        emit(kind="cubical_vs_freudenthal", d=64, h=65, max_abs_diff=float((a - b).abs().max()),
             max_abs=float(b.abs().max()), **common)

    # --- WECT averaged over directions -------------------------------------- #
    batch = batch_size(name)
    hmax = max(HEIGHTS)
    sample = sample_directions_2d if dim == 2 else sample_directions_3d

    def record(sampling, d, out, t, mem, b):
        errs = {str(h): rel_err(out[::(hmax - 1) // (h - 1)], reference(ref_name, h)) for h in HEIGHTS}
        emit(kind="wect", sampling=sampling, d=d, batch=b, seconds=t, peak_bytes=mem, rel_err=errs, **common)

    max_d = DIRS[-1]
    for d in DIRS:  # pyECT's own directions: timed
        dirs, b = sample(d, device=GPU), min(batch, d)
        fn = lambda: wect_mean(cx32, dirs, hmax, b)
        if d <= 64:
            t, _, _ = measure(fn, 1)
            t, mem, out = measure(fn, reps_for(t))
        else:
            t, mem, out = measure(fn, 1, warmup=False)
        record("grid", d, out, t, mem, b)
        if t > WECT_TIME_CAP:
            max_d = d
            break
    for d in DIRS:  # i.i.d. uniform directions: error only
        if d > max_d:
            break
        out = wect_mean(cx32, random_directions(d, dim, seed=d), hmax, min(batch, d))
        record("random", d, out, float("nan"), 0, min(batch, d))

    # h-independence of the WECT's cost, at d = 1024
    d = 1024
    dirs = sample(d, device=GPU)
    for h in (HEIGHTS[0], HEIGHTS[-1]):
        t, mem, _ = measure(lambda: wect_mean(cx32, dirs, h, min(batch, d)), 3)
        emit(kind="wect_h_check", d=d, h=h, seconds=t, peak_bytes=mem, **common)

    del cx64, cx32
    if CUDA:
        torch.cuda.empty_cache()

emit(kind="done")
OUT.close()